# Building and Training a Graph Neural Network for Spatial Upscaling with Anemoi-Graphs

**Authors**: Mario Santa Cruz Lopez (ECMWF), adapted by Martin Janssens (WUR)

*This notebook was last tested and operational on 13/05/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was adapted for the DestinE [2026 Machine Learning for Earth System Modelling Course](https://learning.ecmwf.int/course/view.php?id=99). It provides a hands-on introduction to building and training Graph Neural Networks (GNNs) for spatial downscaling using Anemoi-Graphs.
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
This notebook can be run/accessed on the following free online platforms. Please note they are not officially supported by or linked with ECMWF.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/m2/GNNs_anemoi.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/GNNs_anemoi.ipynb)
[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/2026-ml-esm-training/main?binder_path=m2&labpath=m2/GNNs_anemoi.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/GNNs_anemoi.ipynb)
<!--
::: -->

## Introduction

In this session, we will:

* Build a bipartite graph connecting a coarse and a fine reduced Gaussian grid using Anemoi-Graphs.
* Implement a Graph Neural Network (GNN) that passes information from the coarse grid to the fine grid.
* Train the GNN to upscale data, i.e. predict fine-grid values from the coarse grid, on a dummy dataset.
* Visualise the GNN predictions
* Extend the architecture with attention mechanisms and multi-head attention.

By going through this exercise, we will also introduce the core ideas behind graph-based neural weather models, such as those underpinning ECMWF's Artificial Integrated Forecast System (AIFS).

### Why Graph Neural Networks for Meteorological Data?

In previous sessions, you worked with Convolutional Neural Networks (CNNs), which are well-suited for data on regular grids such as images. However, weather and climate data is typically produced on irregular grids (e.g. reduced Gaussian grids, icosahedral meshes, or unstructured observation networks) and CNNs cannot naturally handle this kind of data. Additionally, many atmopsheric processes have spatial dependencies that vary with location on the globe; these cannot be captured by a fixed convolutional kernel.

### The GNN Solution

Graph Neural Networks address these challenges by representing data as a graph: nodes hold the data at each grid point, and edges encode the spatial relationships between points, or between sets of points. The key operation is called message passing: each node gathers information from its neighbours, aggregates it, and updates its own state. This process is:

- Flexible: It works on any graph topology, regular or irregular, and it can easily be adapted based on relationships you hypothesise to be important in your problem.
- Explicit about structure: spatial relationships are encoded directly in the graph connectivity rather than assumed by a kernel shape.
- Scalable: local message passing can be parallelised efficiently across nodes.

In this notebook, we build a *bipartite GNN* that passes messages from a coarse input grid to a fine target grid. While such "upscaling" is itself an interesting task, it is also an operation that frequently occurs in modern neural weather models.

### In this notebook

So, in this notebook, we'll implement a GNN for spatial upscaling (coarse-to-fine interpolation). We will use [Anemoi-Graphs](https://anemoi-graphs.readthedocs.io) to construct the graph and [PyTorch Geometric](https://pytorch-geometric.readthedocs.io) to implement the message passing layers.

The steps we will follow are:

- Creating a bipartite graph between a coarse (o48, ~2°) and a fine (o96, ~1°) reduced Gaussian grid
- Inspecting the graph structure using the Anemoi-Graphs inspector
- Implementing a simple bipartite GNN (`BipartiteGNN`) with sum aggregation
- Training the GNN on a dummy dataset and visualising the results
- Extending the architecture with attention-based message passing (`GraphTransformer`)
- Further extending to multi-head attention (`MultiHeadGraphTransformer`)

## Prepare your environment

The following packages are used in this notebook:

- `torch` and `torch_geometric` for building and training graph neural networks
- `anemoi-graphs` for constructing bipartite graphs on reduced Gaussian grids
- `einops` for readable tensor reshaping
- `numpy` for handling arrays and mathematical functions
- `matplotlib` for visualisation

We begin by making sure these are installed (this may take a little while).

In [1]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m2/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.1/116.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.1 MB/s eta 0:00:00


In [2]:
import einops
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch_geometric.data import HeteroData
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from IPython.display import IFrame

from anemoi.graphs.edges import KNNEdges, CutOffEdges
from anemoi.graphs.edges.attributes import GaussianDistanceWeights
from anemoi.graphs.nodes import ReducedGaussianGridNodes
from anemoi.graphs.inspect import GraphInspector

# Example training loop for DownscalingModel with DataLoader
NUM_EPOCHS = 10
STEPS_PER_EPOCH = 100
HIDDEN_DIM = 16

## Create a simple graph with Anemoi-Graphs

The goal of this notebook is to upscale data, i.e. to interpolate it from a coarse mesh to a fine mesh. These will both be represented by a set of nodes, and they will be connected to each other by edges, which together will form the graph we will work with. So let's start by first creating the graph's nodes using `ReducedGaussianGridNodes` from `anemoi.graphs.nodes`.

### A Reduced Gaussian Grid of graph nodes

A **Reduced Gaussian Grid (RGG)** is the standard spatial grid used by ECMWF's Integrated Forecasting System (IFS). It has two defining properties:

- **Gaussian latitudes**: the latitude rows are placed at the roots of Legendre polynomials rather than at evenly spaced intervals. This placement has advantages for the accuracy of IFS's spectral transforms.
- **Reduced**: instead of using the same number of grid points on every latitude row, the number of points per row decreases towards the poles. This keeps the physical spacing between neighbouring points roughly constant across the globe.

The grids here carry an **"o" prefix** (e.g. `o48`, `o96`), indicating the modern *octahedral* variant, where the reduction in grid points per row follows a pattern based on the octahedron. The number after "o" is the truncation parameter N, which determines the resolution:

| Grid | Approx. resolution | Number of grid points |
|------|-------------------|----------------------|
| o48  | ~2°               | 10,944               |
| o96  | ~1°               | 40,320               |

We create **two sets of nodes** — one for each resolution.

In [3]:
# Create coarse and fine grid nodes
coarse_node_builder = ReducedGaussianGridNodes('o48', name='input')
fine_node_builder = ReducedGaussianGridNodes('o96', name='target')

graph = HeteroData()
graph = coarse_node_builder.update_graph(graph)
graph = fine_node_builder.update_graph(graph)
print(graph)

HeteroData(
  input={
    x=[10944, 2],
    node_type='ReducedGaussianGridNodes',
    _grid_reference_distance=0.03782872513455103,
  },
  target={
    x=[40320, 2],
    node_type='ReducedGaussianGridNodes',
    _grid_reference_distance=0.019504681036259703,
  }
)


### Grid edges

Now that we have our nodes, we need to connect them with edges to define which coarse-grid points can send information to which fine-grid points.

**K-Nearest Neighbours (KNN) edges** connect each target node to its K closest source nodes, measured by great-circle distance on the sphere. With `num_nearest_neighbours=4`, every fine-grid node receives edges from its 4 nearest coarse-grid neighbours. This means the total number of edges is simply 4 × 40,320 = **161,280**, which we can verify in the graph printout below.

KNN connectivity is a common choice in GNN-based weather models for several reasons:

- **Uniform degree**: every target node has exactly K incoming edges, which makes memory allocation and batching predictable.
- **Local support**: each fine-grid point only aggregates information from the few coarse-grid points that are geographically closest to it. This encodes our inductive bias that nearby grid points are most relevant for the interpolation.
- **Bipartite structure**: edges only connect the two node sets (coarse and fine); there are no edges within a single coarse or fine grid. This is the structure a `BipartiteGNN` is designed to operate on.

In [4]:
# Create edges from coarse to fine nodes
edge_builder = KNNEdges(num_nearest_neighbours=4, source_name='input', target_name='target')
# Alternatively, use CutOffEdges
# edge_builder = CutOffEdges(cutoff_factor=0.7, source_name='input', target_name='target')

graph = edge_builder.update_graph(graph)
print(graph)

Installing 'torch-cluster' can significantly improve performance for graph creation.
You can install it using:
    TORCH_VERSION=$(python -c "import torch; print(torch.__version__)")
    pip install torch-cluster -f https://data.pyg.org/whl/torch-${TORCH_VERSION}.html



HeteroData(
  input={
    x=[10944, 2],
    node_type='ReducedGaussianGridNodes',
    _grid_reference_distance=0.03782872513455103,
  },
  target={
    x=[40320, 2],
    node_type='ReducedGaussianGridNodes',
    _grid_reference_distance=0.019504681036259703,
  },
  (input, to, target)={
    edge_index=[2, 161280],
    edge_type='KNNEdges',
  }
)


## Graph Inspection

This section demonstrates how to inspect and visualize the graph structure created with anemoi-graphs.  

- Using `GraphInspector` from `anemoi.graphs.inspect`, you can interactively explore node types, edge connections, and graph attributes. When you run `GraphInspector("graphs/my_first_graph.pt", "interactive_plots/").inspect()`, it creates a set of static PNG and interactive HTML files under the specified folder (`interactive_plots/`). These files allow you to visually inspect the graph structure, node distributions, and edge connections in your browser.
- For command-line inspection, you can use the CLI tool for a summary and visualization directly in the terminal:

    ```bash
    anemoi-graphs inspect graph.pt interactive_plots/
    ```


In [5]:
os.makedirs("graphs", exist_ok=True)
torch.save(graph, "graphs/my_first_graph.pt")
GraphInspector("graphs/my_first_graph.pt", "interactive_plots/").inspect()
plt.close('all')

In [ ]:
# Open the graph, zoom and pan to get some more resolution, and click the target and input nodes to toggle them
from IPython.display import HTML
HTML(open('interactive_plots/input_to_target.html').read())

## Datasets and utility functions

We now define some helper classes and functions for the rest of the notebook.

### Datasets

The following cell defines two dataset classes used throughout the notebook:

- **DummyDataset**: Generates random 2D sine wave fields on the fine (`o96`) grid as synthetic targets, then interpolates them down to the coarse (`o48`) grid using Gaussian-weighted KNN edges to produce the inputs. Each sample is a `(x_coarse, x_fine)` pair ready for training a downscaling model.
    *Usage*:  
    ```python
    dataset = DummyDataset(num_samples=100, graph=graph)
    x_coarse, x_fine = dataset[0]
    ```

- **DownscalingModel**: A thin `nn.Module` wrapper that pairs a GNN with the graph. It exposes the source and target node coordinates as properties and handles the forward pass, passing coarse-grid features and the graph's edge index to the underlying GNN.  
    *Usage*:  
    ```python
    model = DownscalingModel(gnn=gnn, graph=graph)
    prediction = model(x_coarse.unsqueeze(0))
    ```

In [6]:
class DummyDataset(Dataset):
    """Dummy dataset for downscaling task of random 2D sine wave fields."""

    def __init__(self, num_samples: int, graph: HeteroData):
        self.num_samples = num_samples
        self.graph = graph
        self.proj_matrix = self.build_interp_matrix(graph)

    @property
    def num_variables(self):
        return 1

    def build_interp_matrix(self, graph: HeteroData):
        edge_builder = KNNEdges(num_nearest_neighbours=3, source_name="target", target_name="input")
        graph = edge_builder.update_graph(graph)
        weights = GaussianDistanceWeights(norm="l1")(
            x=(graph["target"], graph["input"]), edge_index=graph["target", "to", "input"].edge_index
        )

        interp_matrix = torch.sparse_coo_tensor(
            graph["target", "to", "input"].edge_index,
            weights.squeeze(),
            (graph["target"].num_nodes, graph["input"].num_nodes),
            device=graph["target", "to", "input"].edge_index.device,
        )
        return interp_matrix.coalesce().T

    def _create_random_2d_sine_wave_field(self):
        sine_wave = np.sin(10 * np.random.rand() * self.graph["target"].x[:, 0]) * np.cos(
            10 * np.random.rand() * self.graph["target"].x[:, 1]
        )
        return sine_wave.to(torch.float32).unsqueeze(-1)

    def _interpolate_to_coarse(self, fine_field):
        return torch.sparse.mm(self.proj_matrix.to(fine_field.device), fine_field)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x_fine = self._create_random_2d_sine_wave_field()
        x_coarse = self._interpolate_to_coarse(x_fine)
        return x_coarse, x_fine


class DownscalingModel(nn.Module):
    """A model wrapper around GNNs"""

    def __init__(self, gnn, graph):
        super().__init__()
        self.gnn = gnn
        self.graph = graph

    @property
    def src_coords(self):
        return self.graph["input"].x

    @property
    def dst_coords(self):
        return self.graph["target"].x

    def forward(self, x_src):
        # We suppose batch size = 1 for simplicity
        # x_src: [num_coarse_nodes, in_channels_src]
        # x_dst: [num_fine_nodes, in_channels_dst]
        assert x_src.shape[0] == 1, "Batch size greater than 1 not supported in this example."
        out = self.gnn(
            x_src=x_src[0, ...].to(torch.float32),
            x_dst=self.graph["target"].x.to(torch.float32),
            edge_index=self.graph["input", "to", "target"].edge_index.to(torch.int64),
            edge_attr=None,
        )
        return out


In [7]:
# Create a dummy dataset
dataset = DummyDataset(num_samples=100, graph=graph)

Installing 'torch-cluster' can significantly improve performance for graph creation.
You can install it using:
    TORCH_VERSION=$(python -c "import torch; print(torch.__version__)")
    pip install torch-cluster -f https://data.pyg.org/whl/torch-${TORCH_VERSION}.html

/tmp/ipykernel_2317/1006938095.py:20: UserWarning:

Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)



In [19]:
TORCH_VERSION = torch.__version__
print(TORCH_VERSION)

!pip install torch-cluster -f https://data.pyg.org/whl/torch-${TORCH_VERSION}.html

2.11.0+cu128
Looking in links: https://data.pyg.org/whl/torch-.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 319, in build
    wheel_file = _build_one(
                 ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_inte

### Utility functions
The following cell defines utility functions that streamline model training and evaluation:

- **train()**: Trains a GNN model using a dataset for a specified number of epochs and steps per epoch. It returns the trained model and a list of training losses.  
    *Usage*:  
    ```python
            model, train_losses = train(gnn, dataset, epochs=100, steps_per_epoch=1000)
    ```

- **plot_loss_curve()**: Plots the training loss curve over epochs to visualize model convergence.  
    *Usage*:  
    ```python
            plot_loss_curve(train_losses)
    ```

- **plot_sample()**: Visualizes the input (coarse), target (fine), model prediction, and error for a single sample from the dataset.  
    *Usage*:  
    ```python
            plot_sample(model, dataset[0])
    ```

In [ ]:
def train(
        gnn: nn.Module,
        dataset: Dataset,
        epochs: int,
        steps_per_epoch: int,
        lr: float = 1e-3
    ) -> list[float]:
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
    model = DownscalingModel(gnn=gnn, graph=dataset.graph)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses = []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for i, (x_coarse, x_fine) in enumerate(dataloader):
            optimizer.zero_grad()
            y_pred = model(x_coarse)  # shape: (batch, len_fine, num_vars)
            loss = criterion(y_pred, x_fine)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x_coarse.size(0)
            if i + 1 >= steps_per_epoch:
                break
        epoch_loss /= steps_per_epoch
        train_losses.append(epoch_loss)
        print(f"Epoch {epoch}, Loss: {epoch_loss:.4f}")
    return model, train_losses


# Plotting utility functions
def plot_loss_curve(train_losses):
    plt.figure(figsize=(8, 4))
    plt.semilogy(train_losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title("Training Loss Curve")
    plt.grid(True)
    plt.show()


def plot_sample(model, sample):
    x_coarse, x_fine = sample
    y_pred = model(x_coarse.unsqueeze(0)).detach().cpu()

    plt.figure(figsize=(12, 12))

    # Input
    plt.subplot(2, 2, 1)
    plt.scatter(
        model.src_coords[:, 1],
        model.src_coords[:, 0],
        c=x_coarse.numpy().squeeze(),
        cmap="viridis"
    )
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Input (Coarse)")
    plt.colorbar(label="Value")

    # Target
    plt.subplot(2, 2, 2)
    plt.scatter(model.dst_coords[:, 1], model.dst_coords[:, 0], c=x_fine.numpy().squeeze(), cmap="viridis")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Target (Fine)")
    plt.colorbar(label="Value")

    # Prediction
    plt.subplot(2, 2, 3)
    plt.scatter(model.dst_coords[:, 1], model.dst_coords[:, 0], c=y_pred.numpy().squeeze(), cmap="viridis")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Prediction")
    plt.colorbar(label="Predicted Value")

    # Error
    plt.subplot(2, 2, 4)
    plt.scatter(model.dst_coords[:, 1], model.dst_coords[:, 0], c=(y_pred - x_fine).numpy().squeeze(), cmap="coolwarm")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Error (Prediction - Target)")
    plt.colorbar(label="Error")

    plt.tight_layout()
    plt.show()


## Build your first Graph Neural Network

Now, let's build a simple Graph Neural Network using PyTorch Geometric, to take care of the upscaling for us. The core of a GNN in PyTorch Geometric is the `MessagePassing` class, which abstracts the process of exchanging information between nodes. Its workflow is structured around four key methods: `propagate()`, `message()`, `aggregate()` and `update()`.

Before covering these functions in detail, let's get an overview of what we have, and want to do. We want to send information from a set of source nodes (denoted by `_src`, here the coarse grid nodes) to the target nodes (`_dst`, here fine grid nodes). On each of these sets of nodes, we will have a number of meteorological variables (e.g. temperature, wind, pressure, etc.), here referred to as `in_channels_src` (how many variables are at the source nodes) and `out_channels` (the same dimension, but now at the target nodes). That is, we will want to transform between these two 2D tensors:
- `x_src`, shape `(num_src_nodes, in_channels_src)`
- `x_dst`, shape `(num_dst_nodes, out_channels)`


**message()**:
The first task is now to create a message, to be sent along each edge from a source (coarse) node to a destination (fine) node. This is achieved by the `message()` function. It does the following (see summary diagram below):

  1. Create the edge input `x_j`. `x_j` basically contains the same information as the input feature matrix `x_src`, with a slight difference: A single coarse node can be the nearest neighbour of many fine-grid nodes simultaneously, so to pass a message between each coarse node and all its nearest neighbours on the fine grid, `x_src` is expanded by copying each set of `in_channels_src` features at a coarse node to all the edges that need connecting, giving `x_j` the dimension `(num_edges, in_channels_src)`.
  2. Projection (in the cell below, `self.lin_src(x_j)`): This projects each edge's source features from `in_channels_src` into an intermediate layer of size `hidden_dim`, giving a matrix of shape `(num_edges, hidden_dim)`. This will be a learned transformation into the network's internal (hidden) representation space. The `hidden_dim` is not tied to any physical variable. It represents how large the space of learned intermediate features can be, much like an intermediate layer in a dense neural network.
  3. Edge attribute addition (optional): Optionally, each edge can have an attribute (e.g. a distance weight). If these are given, they are projected to the same `hidden_dim`, and added element-wise. This may then help the network influence each message by how important an edge is deemed to be.

The function then returns the message `m_j`, with shape `(num_edges, hidden_dim)`, i.e. it creates one message (in the internal space) per edge.

![Message passing](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/assets/gnn-message.png?raw=1)


**aggregate()**: The message `m_j` now exists on each edge. Each each target (fine-grid) node is connected to several of these edges. To determine what to do on such a node, you will usually combine all the incoming messages at each target node into an "aggregated" message, e.g. with a sum, mean, max of the incoming messages, at each target node (see the upper branch of the diagram below). So if the aggregation method is `sum`, and if a fine-grid node receives 4 messages, each of size `hidden_dim`, they are summed *element-wise* into one vector of size `hidden_dim`. The output of `aggregate()` is therefore a tensor of shape `(num_dst_nodes, hidden_dim)` — one aggregated message vector per fine-grid node. In the cell below, the aggregation method of the `MessagePassing` class is set via the `aggr` argument in the constructor.

    *Example*:  
    ```python
    super().__init__(aggr='add')  # Use sum aggregation
    # No need to override aggregate() unless you want a custom behavior
    ```
![GNN Layer Structure Schema](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/assets/agg_update_schema.png?raw=1)

**update()**: Updates the aggregated message of shape `(num_dst_nodes, hidden_dim)` into the final output `x_dst` of shape `(num_dst_nodes, out_channels)`. It could just do this by directly applying another linear transformation (`self.projection` in the cell below) to the aggregated message. However, to predict the fine-grid state, we may wish to use each fine-grid node's specific location, and not just the aggregated information from its neighbours. For example, we know the precise latitude and longitude of these points, and we might know things like the orography, surface-cover type etc. Therefore, one usually also passes such information into the update.

Here, this is done by considering a part of `x_dst`, which has shape `(num_dst_nodes, in_channels_dst)`. Here, `in_channels_dst` represents the number of variables we already know at the the target grid points, in this case just the latitude and the longtiude). This tensor is then passed through another linear layer (`self.lin_dst` in the cell below), to turn its dimension into `(num_dst_nodes, hidden_dim)`, the same dimension as the aggregated message.

Then, the aggregated message and the transformed `x_dst` are added together element-wise (see the diagram above), before a final linear transformation takes care of the final prediction of the `out_channels` meteorological variables at the target nodes. This layer must thus transform the total message into the final, desired `x_dst` of shape `(num_dst_nodes, out_channels)`.

    *Example*:  
    ```python
    def update(self, aggr_out, x: tuple):
        x_dst = self.lin_dst(x[1]) # Transform the known information at the target nodes x[1] into the hidden_dim
        return self.projection(aggr_out + x_dst)
    ```

By customizing these methods, you can implement a wide variety of GNN architectures tailored to your graph data and task.

<!-- Below are two example schema diagrams illustrating the structure of the message passing workflow in a GNN. These images help visualize how node features and edge attributes flow through the `propagate`, `message`, `aggregate`, and `update` methods. -->


<!--
class BipartiteGNN(MessagePassing):
    def __init__(
        self,
        in_channels_src: int,
        in_channels_dst: int,
        hidden_dim: int,
        out_channels: int,
        edge_attr_dim: int = 0,
    ):
        super().__init__(aggr='add')
        self.lin_src = torch.nn.Linear(in_channels_src, hidden_dim)
        self.lin_dst = torch.nn.Linear(in_channels_dst, hidden_dim)
        self.lin_edges = torch.nn.Linear(edge_attr_dim, hidden_dim) if edge_attr_dim > 0 else None
        self.projection = nn.Linear(hidden_dim, out_channels)

    def forward(self, x_src, x_dst, edge_index, edge_attr=None):
        # x_src: [num_src_nodes, in_channels_src]
        # x_dst: [num_dst_nodes, in_channels_dst]
        # edge_index: [2, num_edges] (from src to dst)
        out = self.propagate(
            x=(x_src, x_dst),
            edge_index=edge_index.to(torch.int64),
            edge_attr=edge_attr
        )
        # out: [num_dst_nodes, hidden_dim]

        out = self.projection(out)
        # out: [num_dst_nodes, out_channels]

        return out

    def message(self, x_j, edge_attr=None):
        # x_j: source node features
        # x_j: [num_edges, in_channels_src]

        m_j = self.lin_src(x_j)
        # m_j: [num_edges, hidden_dim]

        if edge_attr is not None:
            edge_attr = self.lin_edges(edge_attr)
            # edge_attr: [num_edges, hidden_dim]

            m_j = m_j + edge_attr

        return m_j

    def update(self, aggr_out, x: tuple):
        # x: tuple (x_src, x_dst)

        x_dst = self.lin_dst(x[1])
        # x_dst: [num_dst_nodes, hidden_dim]

        return aggr_out + x_dst
-->

### Task: Create a Bipartite GNN class with the right dimensions
Fill in the blanks indicated by `???` in the `toch.nn.Linear` layers in the `BipartiteGNN` class below, so that the linear layers have the correct input and output dimensions. Hint: Use the arguments `in_channels_src`, `in_channels_dst`, `hidden_dim`, and `out_channels` to ensure the shapes match the message passing workflow described above, and the [pytorch docs of the `torch.nn.Linear` function](https://docs.pytorch.org/docs/2.11/generated/torch.nn.Linear.html).

- `self.lin_src = torch.nn.Linear(in_channels_src, ???)`
- `self.lin_dst = torch.nn.Linear(???, hidden_dim)`
- `self.projection = nn.Linear(???, ???)`

Try to reason about the shapes at each step and refer to the schema diagrams and explanation above for guidance. Once you have filled in the blanks, run the cell to check if your implementation works!

In [ ]:
class BipartiteGNN(MessagePassing):
    def __init__(
        self,
        in_channels_src: int,
        in_channels_dst: int,
        hidden_dim: int,
        out_channels: int,
        edge_attr_dim: int = 0,
    ):
        super().__init__(aggr='add')
        self.lin_src = torch.nn.Linear(in_channels_src, ???)
        self.lin_dst = torch.nn.Linear(???, hidden_dim)
        self.lin_edges = torch.nn.Linear(edge_attr_dim, hidden_dim) if edge_attr_dim > 0 else None
        self.projection = nn.Linear(???, ???)

    def forward(self, x_src, x_dst, edge_index, edge_attr=None):
        # x_src: [num_src_nodes, in_channels_src]
        # x_dst: [num_dst_nodes, in_channels_dst]
        # edge_index: [2, num_edges] (from src to dst)
        out = self.propagate(
            x=(x_src, x_dst),
            edge_index=edge_index.to(torch.int64).to(x_src.device),
            edge_attr=edge_attr
        )
        # out: [num_dst_nodes, hidden_dim]

        out = self.projection(out)
        # out: [num_dst_nodes, out_channels]

        return out

    def message(self, x_j, edge_attr=None):
        # x_j: source node features
        # x_j: [num_edges, in_channels_src]

        m_j = self.lin_src(x_j)
        # m_j: [num_edges, hidden_dim]

        if edge_attr is not None:
            edge_attr = self.lin_edges(edge_attr)
            # edge_attr: [num_edges, hidden_dim]

            m_j = m_j + edge_attr

        return m_j

    def update(self, aggr_out, x: tuple):
        # x: tuple (x_src, x_dst)

        x_dst = self.lin_dst(x[1])
        # x_dst: [num_dst_nodes, hidden_dim]

        return aggr_out + x_dst

## Train the GNN on Dummy Data

Let's train the model for a few epochs and observe the loss. This is a demonstration with the dummy data we created before.

In [ ]:
gnn = BipartiteGNN(
    in_channels_src=dataset.num_variables,
    in_channels_dst=2,
    hidden_dim=HIDDEN_DIM,
    out_channels=dataset.num_variables,
    edge_attr_dim=0,
)
model, train_losses = train(gnn, dataset, epochs=NUM_EPOCHS, steps_per_epoch=STEPS_PER_EPOCH)


Now plot the loss curve to verify the training procedure has gone as expected.

In [ ]:
plot_loss_curve(train_losses)

And finally inspect the predictions, to check if it has done what we expected. Note that this is not really an independent validation of the training, since we just evaluate the model on the same data it was trained on. This tells us that the model has learned something, but it doesn't tell us whether it generalises to other input/output field combinations.

In [ ]:
plot_sample(model, dataset[0])

## Bonus I: Attentional convolution, layer normalisation and activation functions

**Note: you may consider waiting to go through this section after having completed the module on transformers, since we have not yet covered attentional convolutions.**

In this section, we
- Implement attentional convolution
- Increase the capacity of the projection layer, by adding `LayerNorm` and `ReLU` to the projection layer.
- Include a `mlp_hidden_ration` argument to increase the hidden dimension of the projection MLP.

Before computing attention, the node features are first normalized using `LayerNorm` to stabilize training and improve convergence. Layer normalization helps ensure that the input distributions to the attention mechanism remain consistent, which is especially important when stacking multiple layers or working with varying graph structures. After normalization, the source and target node features are projected into separate query, key, and value spaces using linear layers. These projections form the basis for computing attention scores and aggregating information from neighboring nodes. This is depicted in the sketch below.
![Graph Attention Procesing](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/assets/graphattention_process.png?raw=1)

The message creation in attentional convolution involves three main steps:

1. **Projection**:  
    - Source node features are projected into key and value vectors:  

        $ \quad \quad \mathbf{k}_i = W_k \mathbf{x}_i $  
        $ \quad \quad \mathbf{v}_i = W_v \mathbf{x}_i $
    - Target node features are projected into query vectors:  
        $ \quad \quad \mathbf{q}_j = W_q \mathbf{x}_j $

2. **Attention Score Computation**:  
    - For each edge from source node $ i $ to target node $ j $, compute the attention coefficient using the scaled dot-product:  
        $ \quad \quad \alpha_{ij} = \frac{(\mathbf{q}_j \cdot \mathbf{k}_i)}{\sqrt{d}} $

    where $ d $ is the dimensionality of the key/query vectors, `hidden_dim`.

3. **Softmax Normalization and Message Aggregation**:  
    - Normalize the attention coefficients across all neighbors of target node $ j $ using a softmax:
      $ \quad \quad a_{ij} = \mathrm{softmax}_j(\alpha_{ij}) $
    - The message from node $ i $ to node $ j $ is the value vector weighted by the normalized attention:  
      $ \quad \quad \mathbf{m}_{ij} = a_{ij} \mathbf{v}_i $
    - The aggregated message at node $ j $ is the sum over all incoming messages:  
      $ \quad \quad \mathbf{m}_j = \sum_{i} \mathbf{m}_{ij} $

This attentional mechanism allows each node to selectively focus on its most relevant neighbors during message passing.

![Graph Attention Message](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m2/assets/graphattention_message.png?raw=1)

### Task: Create a GraphTransformer with the right dimensions
Fill in the blanks indicated by `???` in the `GraphTransformer` class below, so that the `LayerNorm` and linear layers have the correct dimensions.
  
Recall the attention message-passing workflow:
  - `LayerNorm` layers normalise node features before projection, so their `normalized_shape` must match the number of input features at each node type.
  - Keys and values are both projected from source node features; queries are projected from target node features.

  Use the right size arguments to fill in:
  - `self.layer_norm_attention_src = nn.LayerNorm(normalized_shape=???)` (normalises source (coarse-grid) features)
  - `self.layer_norm_attention_dst = nn.LayerNorm(normalized_shape=???)` (normalises target (fine-grid) features)
  - `self.lin_value = nn.Linear(???, self.hidden_dim)` (projects source features into value vectors)

In [ ]:
class GraphTransformer(MessagePassing):
    def __init__(
        self,
        in_channels_src: int,
        in_channels_dst: int,
        hidden_dim: int,
        out_channels: int,
        edge_attr_dim: int = 0,
        mlp_hidden_ratio: int = 4,
        pre_lnorm: bool = False,
    ):
        super().__init__(aggr='add')
        self.hidden_dim = hidden_dim
        self.mlp_hidden_ratio = mlp_hidden_ratio
        self.pre_lnorm = pre_lnorm

        # Define layers
        self.layer_norm_attention_src = nn.LayerNorm(normalized_shape=???)
        self.layer_norm_attention_dst = nn.LayerNorm(normalized_shape=???)

        self.lin_key = nn.Linear(in_channels_src, self.hidden_dim)
        self.lin_query = nn.Linear(in_channels_dst, self.hidden_dim)
        self.lin_value = nn.Linear(???, self.hidden_dim)

        self.lin_dst = nn.Linear(in_channels_dst, self.hidden_dim)
        self.lin_edge = nn.Linear(edge_attr_dim, self.hidden_dim)

        # self.projection = nn.Linear(self.hidden_dim, out_channels)
        self.projection = nn.Sequential(
            nn.LayerNorm(normalized_shape=self.hidden_dim),
            nn.Linear(self.hidden_dim, self.mlp_hidden_ratio * self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.mlp_hidden_ratio * self.hidden_dim, out_channels),
        )

    def forward(self, x_src, x_dst, edge_index, edge_attr):
        query, key, value, edge_attr = self.get_qkve(x_src, x_dst, edge_attr)
        # query: [num_dst_nodes, hidden_dim]
        # key: [num_src_nodes, hidden_dim]
        # value: [num_src_nodes, hidden_dim]

        out = self.propagate(
            edge_index=edge_index.to(torch.int64).to(x_src.device),
            x=(x_src, x_dst),
            size=(x_src.size(0), x_dst.size(0)),
            edge_attr=edge_attr,
            query=query,
            key=key,
            value=value,
        )
        # out: [num_dst_nodes, hidden_dim]

        out = self.projection(out)
        # out: [num_dst_nodes, out_channels]

        return out

    def get_qkve(self, x_src, x_dst, edge_attr):
        if self.pre_lnorm:
            x_src = self.layer_norm_attention_src(x_src)
            x_dst = self.layer_norm_attention_dst(x_dst)

        query = self.lin_query(x_dst)
        key = self.lin_key(x_src)
        value = self.lin_value(x_src)

        if edge_attr is not None:
            edge_attr = self.lin_edge(edge_attr)

        return query, key, value, edge_attr

    def message(
        self,
        query_i: torch.Tensor,
        key_j: torch.Tensor,
        value_j: torch.Tensor,
        edge_attr: torch.Tensor,
        index: torch.Tensor,
        ptr: torch.Tensor,
        size_i: int,
    ) -> torch.Tensor:
        # query_i, key_j, value_j: [num_edges, hidden_dim]

        if edge_attr is not None:
            key_j = key_j + edge_attr

        # Compute attention coefficients
        alpha = (query_i * key_j).sum(dim=-1) / self.hidden_dim ** 0.5
        alpha = softmax(alpha, index, ptr, size_i)
        # alpha: [num_edges]

        if edge_attr is not None:
            value_j = value_j + self.lin_edge(edge_attr)

        out = value_j# * alpha.view(-1, 1)
        # out: [num_edges, hidden_dim]

        return out

    def update(self, aggr_out, x: tuple):
        # x: tuple (x_src, x_dst)

        x_dst = self.lin_dst(x[1])
        # x_dst: [num_dst_nodes, num_heads x hidden_dim]

        return aggr_out + x_dst


# DEBUG: You can use this code to debug the forward pass
gnn = GraphTransformer(
    in_channels_src=2,
    in_channels_dst=2,
    hidden_dim=16,
    out_channels=6,
    edge_attr_dim=1,
    mlp_hidden_ratio=4,
)
output = gnn(
    x_src=graph['input'].x,
    x_dst=graph['target'].x,
    edge_index=graph[('input', 'to', 'target')].edge_index.to(torch.int64),
    edge_attr=None
)
print(output.shape)  # Should be [num_target_nodes, out_channels]

In [ ]:
gnn = GraphTransformer(
    in_channels_src=dataset.num_variables,
    in_channels_dst=2,
    hidden_dim=HIDDEN_DIM,
    out_channels=dataset.num_variables,
    edge_attr_dim=0,
)
model, train_losses = train(gnn, dataset, epochs=NUM_EPOCHS, steps_per_epoch=STEPS_PER_EPOCH)

In [ ]:
plot_loss_curve(train_losses)

In [ ]:
plot_sample(model, dataset[0])

## Bonus II: Multihead attention and query/key normalisation

In this final exercise, you'll extend the attentional convolution to support multi-head attention and optional query/key normalization. Multi-head attention enables the model to capture diverse patterns by attending to information from multiple representation subspaces. The `qk_norm` option applies normalization to the query and key vectors before computing attention scores, which can improve stability and expressiveness.

### Task: Implement a `MultiHeadGraphTransformer`
Extend your GraphTransformer to support multiple attention heads and optional query/key normalisation. Your implementation should differ from GraphTransformer in the following ways:

- Linear layers: all projection layers `(lin_key, lin_query, lin_value, lin_dst)` should output `num_heads * hidden_dim` instead of `hidden_dim`, so each head gets its own slice of the representation.
 - `message()`: reshape the projected tensors from `(num_edges, num_heads * hidden_dim)` to `(num_edges, num_heads, hidden_dim)` before computing attention scores, compute one attention score per head, weight the value vectors accordingly, then reshape the output back to `(num_edges, num_heads * hidden_dim)`.
  - `qk_norm`: if enabled, apply a `LayerNorm` to the query and key vectors *per head*, before computing attention scores. This can stabilise training.
  - projection: the input to the projection MLP is now `num_heads * hidden_dim`.

Hint: Use einops.rearrange with the pattern "edges (heads vars) -> edges heads vars" to split and merge the head dimension. Then compare your attention score computation with the single-head case — the only difference is the extra head dimension.

In [ ]:
class MultiHeadGraphTransformer(MessagePassing):
    def __init__(
        self,
        in_channels_src: int,
        in_channels_dst: int,
        hidden_dim: int,
        out_channels: int,
        num_heads: int = 1,
        edge_attr_dim: int = 0,
        pre_lnorm: bool = False,
        qk_norm: bool = False,
        mlp_hidden_ratio: int = 4,
    ):
        super().__init__(aggr='add')
        self.hidden_dim = hidden_dim
        self.mlp_hidden_ratio = mlp_hidden_ratio
        self.pre_lnorm = pre_lnorm
        self.num_heads = num_heads
        self.qk_norm = qk_norm

        # Define layers
        self.layer_norm_attention_src = nn.LayerNorm(normalized_shape=in_channels_src)
        self.layer_norm_attention_dst = nn.LayerNorm(normalized_shape=in_channels_dst)

        self.lin_key = nn.Linear(in_channels_src, ???)
        self.lin_query = nn.Linear(in_channels_dst, ???)
        self.lin_value = nn.Linear(in_channels_src, ???)

        self.lin_dst = nn.Linear(in_channels_dst, ???)
        self.lin_edge = nn.Linear(edge_attr_dim, ???)

        if self.qk_norm:
            self.q_norm = nn.LayerNorm(self.hidden_dim)
            self.k_norm = nn.LayerNorm(self.hidden_dim)

        self.projection = nn.Sequential(
            nn.LayerNorm(normalized_shape=???),
            nn.Linear(???, self.mlp_hidden_ratio * self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.mlp_hidden_ratio * self.hidden_dim, out_channels),
        )

    def forward(self, x_src, x_dst, edge_index, edge_attr):
        query, key, value, edge_attr = self.get_qkve(x_src, x_dst, edge_attr)
        # query: [num_dst_nodes, num_heads x hidden_dim]
        # key: [num_src_nodes, num_heads x hidden_dim]
        # value: [num_src_nodes, num_heads x hidden_dim]

        out = self.propagate(
            edge_index=edge_index.to(torch.int64).to(x_src.device),
            x=(x_src, x_dst),
            size=(x_src.size(0), x_dst.size(0)),
            edge_attr=edge_attr,
            query=query,
            key=key,
            value=value,
        )
        # out: [num_dst_nodes, num_heads x hidden_dim]

        out = self.projection(out)
        # out: [num_dst_nodes, out_channels]

        return out

    def get_qkve(self, x_src, x_dst, edge_attr):
        if self.pre_lnorm:
            x_src = self.layer_norm_attention_src(x_src)
            x_dst = self.layer_norm_attention_dst(x_dst)

        query = self.lin_query(x_dst)
        key = self.lin_key(x_src)
        value = self.lin_value(x_src)

        if self.qk_norm:
            query = ???
            key = ???

        if edge_attr is not None:
            edge_attr = self.lin_edge(edge_attr)

        return query, key, value, edge_attr

    def message(
        self,
        query_i: torch.Tensor,
        key_j: torch.Tensor,
        value_j: torch.Tensor,
        edge_attr: torch.Tensor,
        index: torch.Tensor,
        ptr: torch.Tensor,
        size_i: int,
    ) -> torch.Tensor:
        query_i = einops.rearrange(query_i, ???, heads=self.num_heads)
        key_j = einops.rearrange(key_j, ???, heads=self.num_heads)
        value_j = einops.rearrange(value_j, ???, heads=self.num_heads)
        # query_i, key_j, value_j: [num_edges, num_heads, hidden_dim]

        if edge_attr is not None:
            edge_attr = einops.rearrange(edge_attr, ???, heads=self.num_heads)
            key_j = key_j + edge_attr

        # Compute attention coefficients
        alpha = (query_i * key_j).sum(dim=-1) / self.hidden_dim ** 0.5
        alpha = softmax(alpha, index, ptr, size_i)
        # alpha: [num_edges, num_heads]

        if edge_attr is not None:
            value_j = value_j + edge_attr

        out = value_j * alpha.view(???)
        # out: [num_edges, num_heads, hidden_dim]

        out = einops.rearrange(out, ???)
        # out: [num_edges, num_heads x hidden_dim]

        return out

    def update(self, aggr_out, x: tuple):
        # x: tuple (x_src, x_dst)

        x_dst = self.lin_dst(x[1])
        # x_dst: [num_dst_nodes, num_heads x hidden_dim]

        return aggr_out + x_dst


# DEBUG: You can use this code to debug the forward pass
gnn = MultiHeadGraphTransformer(
    in_channels_src=2,
    in_channels_dst=2,
    hidden_dim=16,
    out_channels=6,
    edge_attr_dim=1,
    num_heads=3,
)
output = gnn(
    x_src=graph['input'].x,
    x_dst=graph['target'].x,
    edge_index=graph[('input', 'to', 'target')].edge_index.to(torch.int64),
    edge_attr=None
)
print(output.shape)  # Should be [num_target_nodes, out_channels]

In [ ]:
gnn = MultiHeadGraphTransformer(
    in_channels_src=dataset.num_variables,
    in_channels_dst=2,
    hidden_dim=HIDDEN_DIM,
    out_channels=dataset.num_variables,
    edge_attr_dim=0,
)
model, train_losses = train(gnn, dataset, epochs=NUM_EPOCHS, steps_per_epoch=STEPS_PER_EPOCH)

In [ ]:
plot_loss_curve(train_losses)

In [ ]:
plot_sample(model, dataset[0])